# Atlas companion: Pandas, SQL, and Graph tutorials

This notebook teaches the same release-inspection task in two forms:

1. **Pandas/SQL tutorial** — inspect and query the pinned relational catalog,
   resolve one recording, inspect its exact published files, and load its small
   retinotopy layer.
2. **Graph tutorial** — express those same operations as named functions with
   visible input/output ports, inspect the wiring, run the flow, and use an
   interactive recording selector.

Every example uses the published Zhong et al. release. It does not simulate a
figure or calculate a scientific result.

- [Article main text](https://www.nature.com/articles/s41586-025-09180-y#Sec1) · [Methods](https://www.nature.com/articles/s41586-025-09180-y#Sec9) ·
  [Data availability](https://www.nature.com/articles/s41586-025-09180-y#data-availability) ·
  [Code availability](https://www.nature.com/articles/s41586-025-09180-y#code-availability)
- [Published dataset, version 2](https://doi.org/10.25378/janelia.28811129.v2) ·
  [versioned Figshare API record](https://api.figshare.com/v2/articles/28811129/versions/2)


## Published-figure and method index

Use these links when moving from release mechanics to scientific analysis.

| Published item | Contents stated in the paper's figure caption | Analysis method |
|---|---|---|
| [Figure 1](https://www.nature.com/articles/s41586-025-09180-y#Fig1) | VR task and training timeline; licking; imaging; neural selectivity; cortical distributions and regional fractions | [Neural selectivity](https://www.nature.com/articles/s41586-025-09180-y#Sec20), [retinotopy](https://www.nature.com/articles/s41586-025-09180-y#Sec25) |
| [Figure 2](https://www.nature.com/articles/s41586-025-09180-y#Fig2) | Test-1 stimuli and licking; spatial sequences; coding-direction projections and similarity index | [Coding direction and similarity index](https://www.nature.com/articles/s41586-025-09180-y#Sec21) |
| [Figure 3](https://www.nature.com/articles/s41586-025-09180-y#Fig3) | Novel and adapted stimuli; selective-neuron distributions; coding-direction projections | [Neural selectivity](https://www.nature.com/articles/s41586-025-09180-y#Sec20), [coding direction and similarity index](https://www.nature.com/articles/s41586-025-09180-y#Sec21) |
| [Figure 4](https://www.nature.com/articles/s41586-025-09180-y#Fig4) | Rastermap view and the late-cue-versus-early-cue reward-prediction analysis | [Reward-prediction neurons](https://www.nature.com/articles/s41586-025-09180-y#Sec22) |
| [Figure 5](https://www.nature.com/articles/s41586-025-09180-y#Fig5) | Behavioural learning after natural-image, grating, or no pretraining | [Faster task learning after pretraining](https://www.nature.com/articles/s41586-025-09180-y#Sec7) |

Protocol details are in [imaging acquisition](https://www.nature.com/articles/s41586-025-09180-y#Sec13),
[visual stimuli](https://www.nature.com/articles/s41586-025-09180-y#Sec14), and
[behavioural training](https://www.nature.com/articles/s41586-025-09180-y#Sec17). Processing and inference details are in
[processing of calcium-imaging data](https://www.nature.com/articles/s41586-025-09180-y#Sec19) and
[statistics and reproducibility](https://www.nature.com/articles/s41586-025-09180-y#Sec24).


# Tutorial 1 — inspect and load the release through Pandas and SQL

## 1. Connect to the fixed team workspace

`sql.setup()` creates one `ZhongDB`: its tables are ordinary Pandas DataFrames,
and the same tables are registered in DuckDB. The filesystem module remains
private and is used only when a selected array must be fetched and verified.

In [ ]:
#@title Connect to the release { display-mode: "form" }
import os
import sys

try:
    from google.colab import drive as google_drive
except ImportError:
    pass
else:
    google_drive.mount("/content/drive", force_remount=False)
CODE = os.environ.get(
    "ZHONGDB_CODE",
    "/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace/code",
)
if CODE not in sys.path:
    sys.path.insert(0, CODE)

for name in ("database", "drive", "graph", "sql"):
    sys.modules.pop(name, None)
import drive
import graph

db = drive.setup()
db.schema()

## 2. Read release-level metadata

`db.table(name)` returns a DataFrame copy. `release`, `files`, and `experiments`
are tables rather than custom Python objects.

In [ ]:
release = db.table("release").iloc[0]
dataset_summary = {
    "connected": db.connected,
    "root": release["root"],
    "published_files": len(db.table("files")),
    "published_bytes": int(db.table("files")["size_bytes"].sum()),
    "experiment_labels": len(db.table("experiments")),
    "article_id": int(release["article_id"]),
    "version": int(release["version"]),
    "doi": release.get("doi"),
}
dataset_summary

## 3. Filter recordings by experiment or mouse

The `memberships` table maps experiment labels to physical recording IDs. The
`recordings` table contains one row per acquisition.

In [ ]:
supervised_test1 = db.query("""
    SELECT DISTINCT recording_id
    FROM memberships
    WHERE experiment = 'sup_test1'
    ORDER BY recording_id
""")
tx119_sessions = db.query("""
    SELECT recording_id
    FROM recordings
    WHERE mouse = 'TX119'
    ORDER BY date, block
""")

{
    "sup_test1_recording_ids": supervised_test1["recording_id"].tolist(),
    "TX119_recording_ids": tx119_sessions["recording_id"].tolist(),
}

## 4. Inspect one recording and its exact files

A recording is an ordinary row. `recording_files` is the join table connecting
that row to behavior, full neural, reduced neural, and retinotopy files.

In [ ]:
from IPython.display import Markdown, display

recording_id = "TX119_2023_12_24_1"
recording = db.query(
    "SELECT * FROM recordings WHERE recording_id = ?",
    [recording_id],
).iloc[0]
recording_files = db.query("""
    SELECT layer, filename, figshare_file_id, size_bytes, md5
    FROM recording_files
    WHERE recording_id = ?
    ORDER BY layer, experiment
""", [recording_id])

print(
    recording["recording_id"],
    recording["mouse"],
    recording["date"],
    recording["block"],
    recording["experiments_json"],
    sep="\n",
)
rows = [
    f"| `{row.layer}` | [{row.filename}](https://ndownloader.figshare.com/files/{row.figshare_file_id}) | "
    f"{row.size_bytes:,} | `{row.md5}` |"
    for row in recording_files.itertuples()
]
display(Markdown(
    "| Layer | Exact published file | Bytes | MD5 |\n"
    "|---|---|---:|---|\n" + "\n".join(rows)
))

## 5. Load one small data layer

SQL selects the exact retinotopy catalog row; `db.load(...)` then asks the
filesystem layer to copy, checksum, and open only that file.

In [ ]:
retinotopy_file = db.query("""
    SELECT filename, figshare_file_id, size_bytes, md5
    FROM recording_files
    WHERE recording_id = ? AND layer = 'retinotopy'
""", [recording_id]).iloc[0]
retinotopy = db.load(recording_id, "retinotopy")

{
    "published_file": retinotopy_file["filename"],
    "figshare_file_id": int(retinotopy_file["figshare_file_id"]),
    "bytes": int(retinotopy_file["size_bytes"]),
    "md5": retinotopy_file["md5"],
    "variables": {
        name: {"shape": tuple(value.shape), "dtype": str(value.dtype)}
        for name, value in retinotopy.items()
    },
}

## 6. Load larger layers only when needed

The same `db.load(recording_id, layer, experiment=...)` call handles behavior,
SVD, and full neural data. Behavior needs `experiment=` when a recording belongs
to more than one supplied experiment label.

# Tutorial 2 — express the same task as visible graph nodes

The graph uses the same SQL rows and `db.load(...)` call as Tutorial 1.

In [ ]:
@graph.node(outputs="recording")
def resolve_recording(recording_id="TX119_2023_12_24_1"):
    return db.query(
        "SELECT * FROM recordings WHERE recording_id = ?",
        [recording_id],
    ).iloc[0].to_dict()


@graph.node(outputs="file_inventory")
def summarize_files(recording):
    return db.query("""
        SELECT layer, filename, figshare_file_id, size_bytes, md5
        FROM recording_files
        WHERE recording_id = ?
        ORDER BY layer, experiment
    """, [recording["recording_id"]]).to_dict("records")


@graph.node(outputs="retinotopy")
def load_retinotopy(recording):
    return db.load(recording["recording_id"], "retinotopy")


@graph.node(outputs="retinotopy_inventory")
def summarize_retinotopy(retinotopy):
    return {
        name: {"shape": tuple(value.shape), "dtype": str(value.dtype)}
        for name, value in retinotopy.items()
    }


atlas_flow = graph.Graph(
    "Inspect one released recording",
    resolve_recording,
    summarize_files,
    load_retinotopy,
    summarize_retinotopy,
)

## 1. Inspect the graph before running it

`describe()` returns a plain dictionary of nodes, input ports, output ports,
settings, and connections. `diagram()` renders that same contract. Neither call
loads a release file.


In [ ]:
atlas_flow.describe()


In [ ]:
atlas_flow.diagram()


## 2. Run the graph and inspect named outputs

`run()` executes the nodes once in their declared order. The returned `Run`
keeps every named port, the actual setting, execution order, and per-node
timings. This run loads the same small retinotopy file used in Tutorial 1.


In [ ]:
atlas_run = atlas_flow.run()

{
    "settings": dict(atlas_run.settings),
    "execution_order": atlas_run.order,
    "seconds_by_node": dict(atlas_run.timings),
    "exact_files": atlas_run["file_inventory"],
    "retinotopy_variables": atlas_run["retinotopy_inventory"],
}


## 3. Use the same graph interactively

The dropdown values come from a SQL query over the `recordings` table.

In [ ]:
tx119_recording_ids = db.query("""
    SELECT recording_id
    FROM recordings
    WHERE mouse = 'TX119'
    ORDER BY date, block
""")["recording_id"].tolist()
atlas_panel = atlas_flow.widget(
    controls={"recording_id": tx119_recording_ids},
    show="retinotopy_inventory",
)
atlas_panel

## Authors' analysis-code index

The tutorials above stop at data discovery and loading. For scientific
analysis, use the authors' `paper` branch and the corresponding Methods section:

- [d-prime definition](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L370-L374)
- [selectivity threshold, cortical masking, coordinate transform, and density normalization](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L394-L416)
- [frame selection and per-neuron selectivity calculation](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L418-L441)
- [held-out sorting and display-trial split for sequence plots](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L599-L703)
- [ten-fold reward-response selection and held-out evaluation](https://github.com/MouseLand/zhong-et-al-2025/blob/paper/utils.py#L814-L882)

The corresponding paper methods are
[neural selectivity](https://www.nature.com/articles/s41586-025-09180-y#Sec20),
[coding direction and similarity index](https://www.nature.com/articles/s41586-025-09180-y#Sec21), and
[reward-prediction neurons](https://www.nature.com/articles/s41586-025-09180-y#Sec22).


## Continue with the analysis notebooks

- `archived/03_dataset_walkthrough_colab.ipynb` inspects selected released arrays.
- `archived/04_paper_companion_colab.ipynb` follows the paper's experimental sequence.
- `archived/05_reward_dprime_dynamics_colab.ipynb` and
  `archived/06_within_session_dprime_colab.ipynb` address reward-related analyses.

Interpret any computed result against the relevant published figure and method
above, the paper's [Statistics and reproducibility](https://www.nature.com/articles/s41586-025-09180-y#Sec24), and the
exact release-file records used by that computation.
